## EQUIPE

**Integrante 01:** Guilherme Araujo dos Santos — RA: 11202020294

**Integrante 02:** Ian Victor Toniolo Silva — RA: 11202020351


## GRANDE MODELO DE LINGUAGEM (Large Language Model - LLM)

**LLM:** Claude Haiku 5.5 (Anthropic) — identificador na API: `claude-haiku-5-5`

**Link para a documentação oficial:** https://docs.claude.com/en/docs/about-claude/models/overview

## API (Opcional)

**API:** MediaWiki Action API (Wikipédia em português)

**Site oficial:** https://www.mediawiki.org

**Link para a documentação oficial:** https://www.mediawiki.org/wiki/API:Main_page

## IMPLEMENTAÇÃO

### Visão geral
O usuário informa um **tema**. O notebook:
1. Busca o artigo mais relevante na **Wikipédia** via **MediaWiki Action API** (`list=search` + `prop=extracts`);
2. Usa **LangChain** + **Claude Haiku 5.5** para aplicar três técnicas de PLN sobre o texto do artigo:
   - **Sumarização de Textos**
   - **Reconhecimento de Entidades Nomeadas (NER)**, com saída estruturada (Pydantic)
   - **Sistema de Perguntas e Respostas** baseado no artigo (o modelo responde só com o que está no texto recuperado — uma forma simples de RAG)

### 1. Instalação das dependências

In [1]:
# langchain-anthropic: integração do LangChain com os modelos Claude
# requests: chamadas HTTP para a MediaWiki API
# python-dotenv: lê a chave de um arquivo .env quando o notebook roda fora do Colab
%pip install -q langchain langchain-anthropic requests pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 2. Configuração da chave da API da Anthropic
A chave **não** fica no código.
- **No Colab:** clique no ícone de chave (🔑 *Secrets*) na barra lateral, crie um segredo chamado `ANTHROPIC_API_KEY` com a sua chave do Claude Console e habilite o acesso do notebook.
- **Localmente (VS Code/Jupyter):** crie um arquivo `.env` na pasta do projeto com a linha `ANTHROPIC_API_KEY=...`. Ele está no `.gitignore` e não vai para o repositório.

In [2]:
import os

try:
    from google.colab import userdata  # rodando no Colab
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    from dotenv import load_dotenv  # rodando localmente
    load_dotenv()

print("Chave carregada:", bool(os.environ.get("ANTHROPIC_API_KEY")))

Chave carregada: True


### 3. Coleta do córpus: MediaWiki Action API
Duas chamadas ao endpoint `https://pt.wikipedia.org/w/api.php`:
- `action=query&list=search` → encontra o título do artigo mais relevante para o tema;
- `action=query&prop=extracts&explaintext=1` → baixa o conteúdo do artigo em texto puro.

A Wikimedia exige um cabeçalho `User-Agent` identificando o cliente.

In [3]:
import requests

WIKI_API = "https://pt.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "PLN-UFABC-ProjetoPratico/1.0 (projeto academico)"}


def consultar_api(params: dict) -> dict:
    """Chama a MediaWiki API e devolve o JSON; erros da API viram exceção com a mensagem original."""
    resp = requests.get(WIKI_API, params={**params, "format": "json"}, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    dados = resp.json()
    if "error" in dados:
        erro = dados["error"]
        raise RuntimeError(f"MediaWiki API: {erro.get('code')}: {erro.get('info')}")
    if "query" not in dados:
        raise RuntimeError(f"Resposta inesperada da MediaWiki API: {str(dados)[:500]}")
    return dados["query"]


def buscar_titulo(tema: str) -> str:
    """Retorna o título do artigo da Wikipédia mais relevante para o tema."""
    resultados = consultar_api({
        "action": "query",
        "list": "search",
        "srsearch": tema,
        "srlimit": 1,
    })["search"]
    if not resultados:
        raise ValueError(f"Nenhum artigo encontrado para: {tema}")
    return resultados[0]["title"]


def obter_texto(titulo: str, max_chars: int = 15000) -> str:
    """Baixa o texto puro do artigo (limitado a max_chars para controlar custo/tokens)."""
    paginas = consultar_api({
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "titles": titulo,
        "redirects": 1,
    })["pages"]
    texto = next(iter(paginas.values())).get("extract", "")
    return texto[:max_chars]


# Teste rápido
titulo = buscar_titulo("Universidade Federal do ABC")
artigo = obter_texto(titulo)
print("Artigo:", titulo)
print("Tamanho:", len(artigo), "caracteres\n")
print(artigo[:800])

Artigo: Universidade Federal do ABC
Tamanho: 15000 caracteres

A Universidade Federal do ABC (UFABC) é uma instituição pública federal de ensino superior no ABC Paulista. Ocupa o 1° lugar entre as universidades brasileiras no Ranking SCImago nos quesitos “Excelência em Pesquisa”, “Publicações de alta qualidade” e “Impacto normalizado das suas publicações”. Foi avaliada pelo Índice Geral de Cursos (IGC) do MEC como a melhor universidade do Estado de São Paulo, sendo avaliada como a 1ª no ranking de cursos de graduação entre todas as universidades do Brasil. O IGC leva em consideração em sua avaliação fatores como infraestrutura, corpo docente, e nota dos formandos no ENADE. Ocupa o 1º lugar entre as universidades brasileiras no quesito "Internacionalização" no Ranking Universitário Folha 2019.
O presidente da comissão que formulou a proposta da univers


### 4. LLM via LangChain
`ChatAnthropic` é o wrapper do LangChain para os modelos Claude. O Claude Haiku 5.5 não aceita o parâmetro `temperature` (a API devolve erro 400), então usamos os valores padrão de amostragem. `max_tokens=4096` dá folga para o raciocínio interno do modelo, que conta dentro desse limite, e para a saída estruturada do NER.

In [4]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatAnthropic(model="claude-haiku-5-5", max_tokens=4096)

# Teste de conexão
print(llm.invoke("Responda em uma frase: o que é PLN?").content)

PLN (Processamento de Linguagem Natural) é o campo da inteligência artificial que permite que computadores compreendam, interpretem e gerem textos e falas humanas.


### 5. Técnica 1: Sumarização de Textos
Uma *chain* LCEL (`prompt | llm | parser`): o prompt recebe o artigo e pede um resumo em português, em tópicos.

In [5]:
prompt_resumo = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente que resume artigos enciclopédicos em português do Brasil, de forma fiel ao texto."),
    ("human", "Resuma o artigo abaixo em até 5 tópicos curtos.\n\nTítulo: {titulo}\n\nArtigo:\n{artigo}"),
])

chain_resumo = prompt_resumo | llm | StrOutputParser()

resumo = chain_resumo.invoke({"titulo": titulo, "artigo": artigo})
print(resumo)

- **Identificação:** A UFABC é uma instituição pública federal de ensino superior no ABC Paulista, criada pela Lei nº 11.145, de 2005. Seus campi ficam em Santo André (sede) e São Bernardo do Campo.
- **Destaques:** Ocupa o 1º lugar entre as universidades brasileiras no Ranking SCImago em pesquisa e publicações, e no quesito "Internacionalização" do Ranking Folha 2019. Foi avaliada como a melhor universidade paulista no IGC. É a única federal brasileira com 100% de professores doutores.
- **Ensino:** Oferece bacharelados interdisciplinares em quadrimestres, como o Bacharelado em Ciência e Tecnologia e o Bacharelado em Ciências e Humanidades. Após o ciclo básico, o aluno pode escolher entre 21 cursos de graduação. Em 2024 foram aprovados três novos cursos de licenciatura, e o curso de Ciência de Dados começa em 2025.
- **Pesquisa:** Em 2007, a CAPES aprovou 10 dos 11 pedidos de cursos de pós-graduação stricto sensu da universidade. Em 2009, foi a terceira instituição que mais apresentou

### 6. Técnica 2: Reconhecimento de Entidades Nomeadas (NER)
Usamos `with_structured_output` com um esquema **Pydantic**: o LangChain força o modelo a devolver uma lista de entidades já tipadas (PESSOA, ORGANIZAÇÃO, LOCAL, DATA, OUTRO), pronta para virar tabela.

In [6]:
from typing import List, Literal
from pydantic import BaseModel, Field
import pandas as pd


class Entidade(BaseModel):
    texto: str = Field(description="A entidade exatamente como aparece no texto")
    tipo: Literal["PESSOA", "ORGANIZACAO", "LOCAL", "DATA", "OUTRO"]


class ListaEntidades(BaseModel):
    entidades: List[Entidade] = Field(description="Entidades nomeadas distintas encontradas no texto")


prompt_ner = ChatPromptTemplate.from_messages([
    ("system", "Você é um sistema de Reconhecimento de Entidades Nomeadas. Extraia as entidades mais relevantes do texto, sem repetições."),
    ("human", "Extraia até 25 entidades nomeadas do texto abaixo.\n\n{artigo}"),
])

chain_ner = prompt_ner | llm.with_structured_output(ListaEntidades)

resultado_ner = chain_ner.invoke({"artigo": artigo})
df_entidades = pd.DataFrame([e.model_dump() for e in resultado_ner.entidades])
df_entidades.sort_values("tipo").reset_index(drop=True)

,texto,tipo
0,Portugal,LOCAL
1,Santo André,LOCAL
2,ABC Paulista,LOCAL
3,Grande ABC,LOCAL
4,Estado de São Paulo,LOCAL
5,São Bernardo do Campo,LOCAL
6,Centro de Física do Porto,ORGANIZACAO
7,Nano Letters,ORGANIZACAO
8,Le Monde,ORGANIZACAO
9,FAPESP,ORGANIZACAO


In [7]:
# Contagem de entidades por tipo
df_entidades["tipo"].value_counts()

tipo
ORGANIZACAO    26
OUTRO          17
PESSOA          9
LOCAL           6
Name: count, dtype: int64

### 7. Técnica 3: Sistema de Perguntas e Respostas
O artigo recuperado da API é injetado no prompt como **contexto**, e o modelo é instruído a responder **apenas** com base nele (Geração Aumentada por Recuperação simples). Se a resposta não estiver no artigo, ele deve dizer isso em vez de inventar.

In [8]:
prompt_qa = ChatPromptTemplate.from_messages([
    ("system",
     "Responda à pergunta usando SOMENTE o contexto fornecido (um artigo da Wikipédia). "
     "Se a resposta não estiver no contexto, responda: 'O artigo não traz essa informação.' "
     "Responda em português, de forma direta."),
    ("human", "Contexto:\n{artigo}\n\nPergunta: {pergunta}"),
])

chain_qa = prompt_qa | llm | StrOutputParser()

perguntas = [
    "Em que ano a universidade foi criada?",
    "Em quais cidades ficam os campi?",
    "Quem ganhou a Copa do Mundo de 2002?",  # fora do contexto: deve recusar
]

for p in perguntas:
    print("P:", p)
    print("R:", chain_qa.invoke({"artigo": artigo, "pergunta": p}), "\n")

P: Em que ano a universidade foi criada?


R: A UFABC foi criada em **2005**. A lei nº 11.145, que criou a Fundação Universidade Federal do ABC, foi sancionada em 26 de julho de 2005 e publicada no Diário Oficial da União em 27 de julho do mesmo ano. 

P: Em quais cidades ficam os campi?


R: Os campi da UFABC estão sediados nos municípios de Santo André (sede) e São Bernardo do Campo. 

P: Quem ganhou a Copa do Mundo de 2002?


R: O artigo não traz essa informação. 



### 8. Pipeline completo: qualquer tema
Junta tudo em uma função: tema → Wikipédia → resumo + entidades + respostas.

In [9]:
def analisar_tema(tema: str, perguntas: list[str]):
    titulo = buscar_titulo(tema)
    artigo = obter_texto(titulo)
    print(f"=== Artigo: {titulo} ({len(artigo)} caracteres) ===\n")

    print("--- RESUMO ---")
    print(chain_resumo.invoke({"titulo": titulo, "artigo": artigo}), "\n")

    print("--- ENTIDADES NOMEADAS ---")
    ents = chain_ner.invoke({"artigo": artigo}).entidades
    display(pd.DataFrame([e.model_dump() for e in ents]))

    print("\n--- PERGUNTAS E RESPOSTAS ---")
    for p in perguntas:
        print("P:", p)
        print("R:", chain_qa.invoke({"artigo": artigo, "pergunta": p}), "\n")


analisar_tema("Alan Turing", [
    "Onde Alan Turing nasceu?",
    "Qual foi a contribuição dele durante a Segunda Guerra Mundial?",
])

=== Artigo: Alan Turing (15000 caracteres) ===

--- RESUMO ---


- **Quem foi:** Matemático, cientista da computação, lógico, criptoanalista e filósofo britânico (1912–1954), considerado um dos pais da ciência da computação teórica e da inteligência artificial. Formalizou os conceitos de algoritmo e computação com a máquina de Turing.
- **Segunda Guerra Mundial:** Trabalhou em Bletchley Park, centro britânico de criptoanálise, onde liderou a seção de análise da marinha alemã (Hut 8) e desenvolveu técnicas e uma máquina eletromecânica para quebrar as cifras da Enigma. Estima-se que esse trabalho tenha encurtado a guerra na Europa em mais de dois anos.
- **Trabalho após a guerra:** Projetou um dos primeiros computadores de programa armazenado (Mecanismo de Computação Automática), ajudou a desenvolver os computadores de Manchester e interessou-se por biologia matemática, escrevendo sobre as bases químicas da morfogênese.
- **Perseguição e morte:** Foi processado em 1952 por atos homossexuais, aceitou castração química como alternativa à prisão e morreu

,texto,tipo
0,Alan Mathison Turing,PESSOA
1,Londres,LOCAL
2,Wilmslow,LOCAL
3,Cheshire,LOCAL
4,Lei de Segredos Oficiais,OUTRO
5,Escola de Código e Cifras do Governo (GC&CS),ORGANIZACAO
6,Bletchley Park,LOCAL
7,Hut 8,ORGANIZACAO
8,Enigma,OUTRO
9,Laboratório Nacional de Física,ORGANIZACAO



--- PERGUNTAS E RESPOSTAS ---
P: Onde Alan Turing nasceu?


R: Alan Turing nasceu em Maida Vale, Londres, em 23 de junho de 1912. 

P: Qual foi a contribuição dele durante a Segunda Guerra Mundial?


R: Durante a Segunda Guerra Mundial, Turing trabalhou em Bletchley Park, o centro britânico de criptoanálise, onde liderou por um tempo a Hut 8, responsável pela análise criptográfica naval alemã. Ele desenvolveu várias técnicas para acelerar a quebra das cifras alemãs, incluindo melhorias no método polonês e uma máquina eletromecânica que podia encontrar configurações da máquina Enigma. Seu trabalho teve papel crucial na decodificação de mensagens interceptadas, que ajudou os Aliados a vencer batalhas decisivas, como a Batalha do Atlântico. Estima-se que isso encurtou a guerra na Europa em mais de dois anos e salvou mais de 14 milhões de vidas. 



### 9. Conclusão
- **API:** a MediaWiki Action API forneceu o córpus (artigos da Wikipédia em português) sob demanda, a partir de qualquer tema.
- **LangChain:** usado para montar as *chains* (prompt → LLM → parser) e para obter saída estruturada com Pydantic.
- **LLM:** o Claude Haiku 5.5 aplicou três técnicas de PLN: sumarização, NER e perguntas e respostas fundamentadas no texto recuperado.